In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, avg, count, sum as ssum, round as sround

spark = SparkSession.builder.appName("HeartDiseaseAnalytics").getOrCreate()
BASE = "hdfs://namenode:8020/data/healthcare"

df = (spark.read.option("header", True).option("inferSchema", True)
      .csv(f"{BASE}/raw/heart.csv"))
print("Raw record count:", df.count())

chol_median = df.filter(col("Cholesterol") > 0).approxQuantile("Cholesterol", [0.5], 0.01)[0]
print("Median cholesterol used:", chol_median)

clean = (df.dropDuplicates()
    .filter(col("RestingBP") > 0)
    .withColumn("Cholesterol_missing", when(col("Cholesterol") == 0, 1).otherwise(0))
    .withColumn("Cholesterol", when(col("Cholesterol") == 0, chol_median)
                               .otherwise(col("Cholesterol")))
    .withColumn("age_group", when(col("Age") < 40, "1. Under 40")
                            .when(col("Age") < 50, "2. 40-49")
                            .when(col("Age") < 60, "3. 50-59")
                            .otherwise("4. 60+"))
    .withColumn("chol_category", when(col("Cholesterol") < 200, "Normal")
                                .when(col("Cholesterol") < 240, "Borderline")
                                .otherwise("High"))
    .withColumn("bp_category", when(col("RestingBP") < 120, "Normal")
                              .when(col("RestingBP") < 140, "Elevated")
                              .otherwise("High")))
clean.cache()
print("Clean record count:", clean.count())

def rate_by(column):
    return (clean.groupBy(column)
            .agg(count("*").alias("patients"),
                 ssum("HeartDisease").alias("disease_cases"),
                 sround(avg("HeartDisease") * 100, 2).alias("disease_pct"))
            .orderBy(col("disease_pct").desc()))

Raw record count: 918
Median cholesterol used: 237.0
Clean record count: 917


In [2]:
for c in ["Sex", "age_group", "ChestPainType", "ExerciseAngina", "ST_Slope", "chol_category", "bp_category"]:
    print("=== Disease rate by", c)
    print(rate_by(c).toPandas().to_string(index=False))
    print()

=== Disease rate by Sex
Sex  patients  disease_cases  disease_pct
  M       724            457        63.12
  F       193             50        25.91

=== Disease rate by age_group
  age_group  patients  disease_cases  disease_pct
     4. 60+       253            185        73.12
   3. 50-59       373            211        56.57
   2. 40-49       211             85        40.28
1. Under 40        80             26        32.50

=== Disease rate by ChestPainType
ChestPainType  patients  disease_cases  disease_pct
          ASY       496            392        79.03
           TA        46             20        43.48
          NAP       202             71        35.15
          ATA       173             24        13.87

=== Disease rate by ExerciseAngina
ExerciseAngina  patients  disease_cases  disease_pct
             Y       371            316        85.18
             N       546            191        34.98

=== Disease rate by ST_Slope
ST_Slope  patients  disease_cases  disease_pct
  

In [7]:
print(clean.count())

917


In [8]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

cat_cols = ["Sex", "ChestPainType", "RestingECG", "ExerciseAngina", "ST_Slope"]
num_cols = ["Age", "RestingBP", "Cholesterol", "FastingBS", "MaxHR", "Oldpeak"]

data = clean.withColumn("label", col("HeartDisease").cast("double"))
train, test = data.randomSplit([0.8, 0.2], seed=42)
print("Train:", train.count(), " Test:", test.count())

indexers = [StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="keep") for c in cat_cols]
encoder = OneHotEncoder(inputCols=[c + "_idx" for c in cat_cols],
                        outputCols=[c + "_ohe" for c in cat_cols])
assembler = VectorAssembler(inputCols=num_cols + [c + "_ohe" for c in cat_cols],
                            outputCol="features")

models = {
  "LogisticRegression": LogisticRegression(featuresCol="features", labelCol="label", maxIter=100),
  "RandomForest": RandomForestClassifier(featuresCol="features", labelCol="label",
                                         numTrees=100, seed=42),
}

auc_eval = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
def mc(metric):
    return MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction",
                                             metricName=metric)

fitted = {}
for name, algo in models.items():
    pipe = Pipeline(stages=indexers + [encoder, assembler, algo])
    model = pipe.fit(train)
    pred = model.transform(test)
    fitted[name] = (model, pred)
    print(f"\n===== {name} =====")
    print("AUC       :", round(auc_eval.evaluate(pred), 4))
    print("Accuracy  :", round(mc("accuracy").evaluate(pred), 4))
    print("Precision :", round(mc("weightedPrecision").evaluate(pred), 4))
    print("Recall    :", round(mc("weightedRecall").evaluate(pred), 4))
    print("F1        :", round(mc("f1").evaluate(pred), 4))
    print(pred.groupBy("label", "prediction").count().orderBy("label", "prediction").toPandas().to_string(index=False))

Train: 728  Test: 189

===== LogisticRegression =====
AUC       : 0.8964
Accuracy  : 0.8148
Precision : 0.8181
Recall    : 0.8148
F1        : 0.8137
 label  prediction  count
   0.0         0.0     67
   0.0         1.0     23
   1.0         0.0     12
   1.0         1.0     87

===== RandomForest =====
AUC       : 0.9052
Accuracy  : 0.8466
Precision : 0.8479
Recall    : 0.8466
F1        : 0.8461
 label  prediction  count
   0.0         0.0     72
   0.0         1.0     18
   1.0         0.0     11
   1.0         1.0     88


In [11]:
rf_model, rf_pred = fitted["RandomForest"]
attrs = rf_pred.schema["features"].metadata["ml_attr"]["attrs"]
names = [n for _, n in sorted((a["idx"], a["name"]) for grp in attrs.values() for a in grp)]
imps = rf_model.stages[-1].featureImportances.toArray()
for n, v in sorted(zip(names, imps), key=lambda x: -x[1])[:10]:
    print(f"{n:30s} {v:.4f}")

(rf_pred.select("Age", "Sex", "ChestPainType", "label", "prediction", "probability")
        .withColumn("probability", col("probability").cast("string"))
        .coalesce(1).write.mode("overwrite").option("header", True)
        .csv(f"{BASE}/results/rf_predictions"))

ST_Slope_ohe_Up                0.2187
ST_Slope_ohe_Flat              0.1383
ChestPainType_ohe_ASY          0.1181
Oldpeak                        0.0819
ExerciseAngina_ohe_Y           0.0818
ExerciseAngina_ohe_N           0.0708
MaxHR                          0.0558
Age                            0.0506
Cholesterol                    0.0284
ChestPainType_ohe_ATA          0.0269


In [12]:
print(rf_pred.select("Age", "Sex", "ChestPainType", "label", "prediction").limit(5).toPandas().to_string(index=False))

 Age Sex ChestPainType  label  prediction
  41   M           ASY    1.0         1.0
  53   M           ATA    1.0         1.0
  54   F           ASY    1.0         1.0
  61   M           ASY    1.0         0.0
  60   M           ASY    1.0         1.0
